# Woche 5 – Explorative Datenanalyse

## 1. Was ist EDA?

**Explorative Data Analysis (EDA)** ist die systematische Untersuchung eines Datensatzes, bevor formale statistische Modellierung oder Hypothesentests durchgeführt werden.

Zentrale Fragen:


```text
Wie sind die Daten verteilt?
Gibt es Missing Values?
Gibt es Duplicates?
Gibt es Outliers?
Welche Variablen hängen zusammen?
Sind Annahmen für spätere Modelle plausibel?
```

---

## 2. Vier Gruppen von EDA-Methoden

EDA wird entlang zweier Dimensionen strukturiert:


```text
                        nicht-grafisch       grafisch

univariat               Kennzahlen           Histogramm
(Werte von nur einer    Mittelwert            Box Plot
Variable)               Median                Bar Chart

multivariat             Korrelation           Scatter Plot
(2 oder mehr Variablen) Kovarianz             Pair Plot
                        Kreuztabelle          Heatmap
```

**Univariat** = eine Variable.  
**Multivariat** = mehrere Variablen.

---

## 3. Datengrundlage: EV Charging Stations

Die Lösung-Notebooks verwenden:


```text
data/ev_charging_stations.csv
```

Der Datensatz besitzt:


```text
2442 Zeilen
15 Spalten
```

Wichtige Variablen:


```text
power_kw
power_type
amperage
voltage
plug_type
city
lat
lon
```

Einlesen im Stil des Repositorys:


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('./data/ev_charging_stations.csv')
print('Shape:', df.shape)
df.head(3)


Datenqualität prüfen:


In [ ]:
print(f"Number of missing values: {df.isnull().sum().sum()}")
df = df.dropna().reset_index(drop=True)

print()
print("Number of duplicated values:",df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)

Im vorhandenen Datensatz gibt es keine fehlenden Werte und keine vollständig doppelten Zeilen.

---

# Nicht-grafische EDA

## 4. Mean, Median und Mode

### Mean

Arithmetischer Mittelwert:


```text
mean = Summe aller Werte / Anzahl Werte
```

### Median

Der mittlere Wert einer sortierten Verteilung.

### Mode

Der häufigste Wert.

Repository-Beispiel:


In [ ]:
print(f"Mean:   {df['power_kw'].mean():.2f} kW")
print(f"Median: {df['power_kw'].median():.2f} kW")
print(f"Mode:   {df['power_kw'].mode()[0]:.2f} kW")

Resultat für `power_kw`:


```text
Mean   ≈ 50.50 kW
Median = 22.00 kW
Mode   = 22.00 kW
```

Interpretation:


```text
Mean >> Median
```

Das deutet auf eine deutlich **rechtsschiefe Verteilung** hin. Viele Ladestationen besitzen 22 kW, während wenige Schnellladestationen sehr hohe Leistungen erreichen und den Mittelwert nach oben ziehen.

---

## 5. Quantile

Ein Quantil teilt eine sortierte Verteilung an einer bestimmten Position.

Wichtige Quantile:


```text
0.25 -> Q1
0.50 -> Median
0.75 -> Q3
```

Repository-Code:


In [ ]:
num_cols = ['power_kw', 'amperage', 'voltage']

df[num_cols].quantile(
    [0.05, 0.25, 0.50, 0.75, 0.95]).round(1)

Für `power_kw`:

| Quantil | Wert |
|---:|---:|
| 5 % | 11 kW |
| 25 % | 22 kW |
| 50 % | 22 kW |
| 75 % | 22 kW |
| 95 % | 225 kW |

Die grosse Differenz zwischen 75 %- und 95 %-Quantil zeigt den langen rechten Tail.

---

## 6. Variance und Standard Deviation

### Variance

Misst die durchschnittliche quadrierte Abweichung vom Mittelwert.

### Standard Deviation

Quadratwurzel der Varianz. Sie besitzt dieselbe Einheit wie die ursprüngliche Variable.

Repository-Code:


In [ ]:
print('Sample variance:')
print(df[num_cols].var().round(2))

print('Sample standard deviation:')
print(df[num_cols].std().round(2))


Für `power_kw` beträgt die Standardabweichung im vorhandenen Datensatz ungefähr:


```text
77.31 kW
```

Die hohe Streuung entsteht durch die Mischung aus normalen AC-Ladepunkten und leistungsstarken DC-Schnellladestationen.

---

## 7. Skewness

**Skewness** beschreibt die Asymmetrie einer Verteilung.

- **Positive (right-skewed)**: long tail to the right, mean > median
- **Negative (left-skewed)**: long tail to the left, mean < median
- **Zero**: symmetric distribution

In [ ]:
skew = df['power_kw'].skew()
print(f'Skewness: {skew:.3f}')


Für `power_kw`:


```text
Skewness ≈ 3.00
```

Damit ist die Verteilung stark rechtsschief.

---

## 8. Kurtosis

**Kurtosis** beschreibt unter anderem, wie stark eine Verteilung durch extreme Werte beziehungsweise schwere Tails geprägt ist.

- **Excess kurtosis > 0**: heavier tails than normal (leptokurtic)
- **Excess kurtosis < 0**: lighter tails than normal (platykurtic)
- **Excess kurtosis ≈ 0**: similar to normal (mesokurtic)

In [ ]:
kurt = df['power_kw'].kurtosis()
print(f'Excess kurtosis: {kurt:.3f}')


Für `power_kw`:


```text
Excess Kurtosis ≈ 8.95
```

Das deutet auf eine heavy-tailed beziehungsweise leptokurtische Verteilung hin.

---

## 9. Kreuztabelle

Eine **Contingency Table** beziehungsweise Kreuztabelle vergleicht Häufigkeiten zweier kategorialer Variablen.

Repository-Beispiel:


In [ ]:
ct = pd.crosstab(
    df['power_type'],
    df['plug_type'],
    margins=True,
    margins_name='Total')

ct

Damit kann beispielsweise untersucht werden, welche Steckertypen bei welchen Ladearten auftreten.

---

## 10. Covariance

**Covariance** beschreibt, ob zwei numerische Variablen gemeinsam steigen oder fallen.


```text
Covariance > 0 -> beide steigen tendenziell gemeinsam
Covariance < 0 -> eine steigt, die andere fällt
Covariance ≈ 0 -> kein klarer linearer gemeinsamer Trend
```

Nachteil: Die Grössenordnung hängt von den Einheiten der Variablen ab.

---

## 11. Correlation

Die **Pearson Correlation** standardisiert die lineare Beziehung auf den Bereich:


```text
-1 <= r <= +1
```

Interpretation:


```text
r nahe +1 -> starke positive lineare Beziehung
r nahe  0 -> schwache lineare Beziehung
r nahe -1 -> starke negative lineare Beziehung
```

Repository-Code:


In [ ]:
corr = df[num_cols].corr(method='pearson').round(3)
corr

Resultat:

| | power_kw | amperage | voltage |
|---|---:|---:|---:|
| `power_kw` | 1.000 | 0.997 | 0.672 |
| `amperage` | 0.997 | 1.000 | 0.680 |
| `voltage` | 0.672 | 0.680 | 1.000 |

`power_kw` und `amperage` sind im Datensatz sehr stark positiv korreliert.

### Zentraler Pitfall

> **Correlation does not imply causation.**

Eine Korrelation beweist keinen kausalen Zusammenhang.

---

# Grafische EDA

## Line Chart

Ein Liniendiagramm stellt Datenpunkte dar, die durch Linien miteinander verbunden sind. Es wird häufig für Zeitreihendaten verwendet.

In [ ]:
import matplotlib.pyplot as plt

# Median der Stromstärke pro Ladeleistung berechnen
amperage_by_power = df.groupby('power_kw')['amperage'].median().sort_index()

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(
    amperage_by_power.index,
    amperage_by_power.values,
    marker="o",
    linewidth=2,
    color="steelblue"
)

ax.set_xlabel("Charging Power (kW)")
ax.set_ylabel("Median Amperage (A)")
ax.set_title("Median Amperage by Charging Power")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 12. Histogram

Ein Histogram zeigt die Verteilung einer numerischen Variable.

Repository-Code:

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df['power_kw'],
        bins=40,
        color='steelblue',
        edgecolor='white')
ax.set_xlabel('Power (kW)')
ax.set_ylabel('Count')
ax.set_title('Distribution of Charging Power (kW)')
plt.tight_layout()
plt.show()

Interpretation im Datensatz:


```text
viele Werte bei 22 kW
wenige sehr hohe Werte bis 400 kW
-> langer Tail nach rechts
```

### Pitfall

Die Anzahl der Bins beeinflusst das Aussehen des Histogramms deutlich.

---

## 13. Box Plot

Ein Box Plot zeigt kompakt:


```text
Q1
Median
Q3
IQR
Whiskers
potenzielle Outliers
```

Der **Interquartile Range** ist:


```text
IQR = Q3 - Q1
```

Repository-Beispiel:


In [ ]:
import seaborn as sns

fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(
    data=df,
    x='power_type',
    y='power_kw',
    ax=ax,
    order=['AC_1_PHASE', 'AC_3_PHASE', 'DC'])

ax.set_xlabel('Power Type')
ax.set_ylabel('Power (kW)')
ax.set_title('Charging Power by Power Type')
plt.tight_layout()
plt.show()

Interpretation:

- DC-Ladestationen besitzen deutlich höhere Leistungen.
- AC-3-Phase-Ladestationen liegen stark um 22 kW konzentriert.

---

## 14. Bar Chart

Ein Bar Chart eignet sich für Häufigkeiten kategorialer Daten.

Top 10 Städte im Repository:


In [ ]:
top10 = df['city'].value_counts().head(10)

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(top10.index[::-1],
        top10.values[::-1],
        color='steelblue',
        edgecolor='white')
ax.set_xlabel('Number of Charging Stations')
ax.set_title('Top 10 Cities by Number of EV Charging Stations')
plt.tight_layout()
plt.show()

---

## 15. QQ Plot

Ein **Quantile-Quantile Plot** vergleicht die Quantile der beobachteten Daten mit einer theoretischen Verteilung, häufig der Normalverteilung.

Repository-Code:


In [ ]:
import scipy.stats as stats

fig, ax = plt.subplots(figsize=(6, 6))
stats.probplot(df['power_kw'], dist='norm', plot=ax)
ax.set_title('QQ Plot — Charging Power (kW)')
plt.tight_layout()
plt.show()


Interpretation:


```text
Punkte ungefähr auf Diagonale -> Normalverteilung plausibel
starke systematische Abweichung -> Normalverteilung unplausibel
```

Bei `power_kw` weichen die Punkte besonders im oberen Bereich deutlich ab. Die Variable ist nicht normalverteilt.

---

## 16. Scatter Plot

Ein Scatter Plot zeigt die Beziehung zwischen zwei numerischen Variablen.

Repository-Beispiel:


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for ptype, grp in df.groupby('power_type'):
    ax.scatter(grp['voltage'],
               grp['amperage'],
               label=ptype,
               alpha=0.5,
               s=30)

ax.set_xlabel('Voltage (V)')
ax.set_ylabel('Amperage (A)')
ax.set_title('Voltage vs. Amperage by Power Type')
ax.legend()
plt.tight_layout()
plt.show()


Dadurch werden unterschiedliche Cluster für AC und DC sichtbar.

---

## 17. Pair Plot

Ein **Pair Plot** zeigt paarweise Beziehungen zwischen mehreren numerischen Variablen.


In [ ]:
num_cols = ['power_kw', 'amperage', 'voltage']

g = sns.pairplot(
    df[num_cols + ['power_type']],
    hue='power_type',
    diag_kind='hist',
    plot_kws={'alpha': 0.4, 's': 15})

g.fig.suptitle(
    'Scatter Plot Matrix — EV Charging Stations',
    y=1.02)
plt.show()


Vorteil: Mehrere Beziehungen und Verteilungen lassen sich gleichzeitig prüfen.

---

## 18. Hexbin Plot

Bei sehr vielen Punkten kann ein Scatter Plot durch Overplotting unübersichtlich werden.

Ein **Hexbin Plot** zählt Beobachtungen in sechseckigen Bereichen.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

hb = ax.hexbin(
    df['voltage'],
    df['amperage'],
    gridsize=25,
    cmap='Blues',
    mincnt=1)

plt.colorbar(hb, ax=ax, label='Count')
ax.set_xlabel('Voltage (V)')
ax.set_ylabel('Amperage (A)')
ax.set_title('Hexbin: Voltage vs. Amperage')
plt.tight_layout()
plt.show()


---

## 19. Correlation Heatmap

Eine Heatmap stellt eine Korrelationsmatrix grafisch dar.


In [ ]:
corr = df[num_cols].corr(method='pearson')

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    corr,
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    center=0,
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    ax=ax)

ax.set_title('Pearson Correlation Heatmap — EV Charging Stations')
plt.tight_layout()
plt.show()


Eine Heatmap erleichtert das schnelle Erkennen starker positiver und negativer Zusammenhänge.

---

# Visualisierungsbibliotheken

## 20. Matplotlib

`matplotlib` ist die grundlegende Python-Bibliothek für statische Visualisierungen.

Die Vorlesung empfiehlt für strukturiertere Grafiken die objektorientierte Schreibweise:


In [ ]:
fig, ax = plt.subplots()
ax.hist(...)


Typische Methoden:


```text
ax.plot()      -> Line Plot
ax.scatter()   -> Scatter Plot
ax.bar()       -> Bar Chart
ax.hist()      -> Histogram
ax.boxplot()   -> Box Plot
```

---

## 21. Seaborn

`seaborn` baut auf matplotlib auf und bietet komfortable statistische Visualisierungen.

Beispiel aus dem Repository:


In [ ]:
import seaborn as sns

sns.set_theme(style='whitegrid')

fig, ax = plt.subplots(figsize=(9, 4))
sns.histplot(
    data=df,
    x='power_kw',
    hue='power_type',
    kde=True,
    bins=40,
    alpha=0.5,
    ax=ax)

plt.tight_layout()
plt.show()


Weitere wichtige Funktionen:


```text
sns.boxplot()
sns.violinplot()
sns.countplot()
sns.scatterplot()
sns.pairplot()
sns.heatmap()
```

---

## 22. Interaktive Visualisierung und Web Apps

Die Vorlesung behandelt zusätzlich:

| Werkzeug | Zweck |
|---|---|
| Plotly | interaktive Grafiken |
| Dash | Dashboards mit Plotly |
| Flask | leichtgewichtiges Web Framework |
| FastAPI | API-/Web-Anwendungen |
| folium | interaktive Karten |
| Streamlit | schnelle Data Apps |
| D3.js / d3blocks | interaktive Webvisualisierung |

Für die grundlegende EDA sind insbesondere `matplotlib` und `seaborn` zentral.

---

# Automated EDA

## 23. Sweetviz

Automatisierte EDA-Tools können einen ersten Überblick über einen Datensatz erzeugen.

Beispiel aus dem Repository:


In [ ]:
import sweetviz as sv

report = sv.analyze(df)
report.show_html(
    'ev_sweetviz_report.html',
    open_browser=False)


Der Report kann unter anderem enthalten:

- Datentypen
- Missing Values
- Verteilungen
- Kennzahlen
- Beziehungen zwischen Variablen

### Pitfall

Automatisierte EDA ersetzt die eigene Interpretation nicht. Auffällige Resultate müssen fachlich geprüft werden.

---

## 24. Eigene Quick-Profile-Funktion

Das Repository zeigt zusätzlich eine kompakte selbst geschriebene Profiling-Funktion:


In [ ]:
def quick_profile(df: pd.DataFrame, max_cols: int = 10) -> None:
    num_cols = df.select_dtypes(include='number').columns[:max_cols]

    print('=== Quick Profile ===')
    print(f'Rows: {len(df):,}  |  Columns: {df.shape[1]}')
    print(f'Missing values:\n{df[num_cols].isnull().sum().to_string()}')
    print(f'\nDuplicates: {df.duplicated().sum()}')

    print('\nDescriptive statistics:')
    display(df[num_cols].describe().round(2))